# تحدي مُرقّم — 3-way majority vote
**My second selected final · Private 0.723 (tied with the stack)**

CPU-only, runs in about a minute. For every word gap, a punctuation mark is emitted when **at least 2 of 3 systems** predict it:

| System | Public |
|---|---|
| legacy system | 0.688 |
| v25 stack generation | 0.681 |
| v26 stack generation | 0.685 |

Plus one rule: the last word of a document must carry a terminal mark (`.` `!` `؟`) — if the vote leaves none, add `.`.

**Inputs:** the competition `test.csv` + the three submission CSVs (attached as a dataset).

Full writeup & code: https://github.com/naz50/muraqqam-challenge-solution

## 1 · Setup and the official tokenizer

Same byte-identical copy of the organizer's tokenizer I used everywhere — the vote operates on *sets of marks per gap*, exactly like the metric does.

In [ ]:
import os, glob
import pandas as pd

WORK_DIR = ('/kaggle/working' if os.path.exists('/kaggle/working') else '.')
VALID = set('.،؟!:؛-')
SYMBOLS = sorted(VALID)
ORDER = ['؟', '!', '.', ':', '؛', '،', '-']
TERM = set('.!؟')

def tok(pred):
    """Official metric tokenizer: (word, trailing gap marks) pairs."""
    pairs, cur, in_word = [], [], False
    for ch in str(pred):
        if ch.isspace():
            if in_word: pairs.append([''.join(cur), []]); cur=[]; in_word=False
            continue
        if ch in VALID:
            if in_word: pairs.append([''.join(cur), [ch]]); cur=[]; in_word=False
            else:
                if pairs: pairs[-1][1].append(ch)
            continue
        if not in_word: in_word=True; cur=[ch]
        else: cur.append(ch)
    if in_word: pairs.append([''.join(cur), []])
    return pairs

def find(pattern):
    hits = []
    for root in ('/kaggle/input', '.'):
        if os.path.exists(root):
            hits += glob.glob(os.path.join(root, '**', pattern), recursive=True)
    return sorted(set(hits), key=lambda p: (len(p), p))


## 2 · Locate and verify the three systems

Every input file is checked row-by-row: same ids as `test.csv`, and perfect word alignment under the official tokenizer. A misaligned voter would silently corrupt the vote, so this asserts loudly instead.

In [ ]:
test_path = (find('test.csv') or [None])[0]
srcs = []
SYSTEMS = [
    ('legacy 0.688', ['submission_old*.csv', 'submission (18)*.csv', 'submission*18*.csv']),
    ('v25',          ['submission_v25*.csv', 'submission v25*.csv', '*v25*.csv']),
    ('v26',          ['submission_v26*.csv', '*v26*.csv']),
]
for name, pats in SYSTEMS:
    p = None
    for pat in pats:
        hits = [h for h in find(pat) if 'sample_submission' not in h]
        if hits: p = hits[0]; break
    assert p, f'system file not found: {name} (patterns tried: {pats})'
    srcs.append(p)
assert test_path, 'competition test.csv not found among the inputs'
print('test:', test_path)
for p in srcs: print('system:', p)

test_df = pd.read_csv(test_path)
systems = []
for p in srcs:
    df = pd.read_csv(p)
    assert len(df) == len(test_df) and (df['id'].values == test_df['id'].values).all(), f'{p}: rows/ids mismatch'
    sets_per_doc = []
    for raw, pred in zip(test_df['text'], df['final_text']):
        pairs = tok(pred)
        assert [w for w, _ in pairs] == str(raw).strip().split(), f'{p}: misalignment'
        sets_per_doc.append([set(c for c in g if c in VALID) for _, g in pairs])
    systems.append(sets_per_doc)
print('all three systems loaded and alignment-verified')


## 3 · The vote

Mark-level majority (≥2 of 3) per gap + the end-of-document terminal rule. I also count how far the result drifts from each voter — a sanity signal that no single system dominates.

In [ ]:
finals, changed = [], [0, 0, 0]
for ti in range(len(test_df)):
    ws = str(test_df['text'].iloc[ti]).strip().split()
    parts = []
    for wi in range(len(ws)):
        votes = [systems[k][ti][wi] for k in range(3)]
        chosen = set(s for s in SYMBOLS if sum(s in v for v in votes) >= 2)
        if wi == len(ws) - 1 and not (chosen & TERM):
            chosen.add('.')
        for k in range(3):
            changed[k] += chosen != votes[k]
        parts.append(ws[wi] + ''.join(s for s in ORDER if s in chosen))
    finals.append(' '.join(parts))

for raw, fin in zip(test_df['text'], finals):
    assert [w for w, _ in tok(fin)] == str(raw).strip().split(), 'word mismatch!'
print(f'diffs vs each voter: legacy={changed[0]} | v25={changed[1]} | v26={changed[2]}')
print(f'alignment check: all {len(finals)} rows valid')


## 4 · Final alignment check and save

In [ ]:
out = os.path.join(WORK_DIR, 'submission.csv')
pd.DataFrame({'id': test_df['id'], 'final_text': finals}).to_csv(out, index=False)
print('saved', out)


---
This vote and my 16-component stack scored **identically (0.723) on the private leaderboard** — a nice confirmation that both decision layers extracted everything the components had to give.